In [1]:
import sys
import os

# 设置你的 main.py 所在目录路径，例如：
project_dir = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"

# 加入到 sys.path（如果尚未添加）
if project_dir not in sys.path:
    sys.path.append(project_dir)

# 检查是否添加成功
print("Updated sys.path:", sys.path)

Updated sys.path: ['/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/scripts', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python39.zip', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9/lib-dynload', '', '/common/home/sl2148/anaconda3/envs/prune_llm/lib/python3.9/site-packages', '/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/']


In [2]:
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from lib.prune import (
    prune_wanda,
    check_sparsity,
    get_mask,
    prune_wandg_set_difference,
)
from lib.model_wrapper import prune_wanda_v2, prune_wandg
from lib.eval import eval_ppl, eval_zero_shot, eval_attack, load_prompt, load_dataset, format_prompt, _safe_generate, extract_answer
    
from vllm import LLM
import argparse
from tqdm import tqdm
from vllm import SamplingParams
from pathlib import Path
from typing import Dict, List
import json
import random

In [11]:
# 📌 设置参数（你原本命令行中给出的内容）
# 构造参数
args = argparse.Namespace(
    model="llama2-7b-chat-hf",
    model_base="llama2-7b-hf",
    seed=0,
    nsamples=2,
    sparsity_ratio=0.5,
    sparsity_type="unstructured",
    prune_method="prune_wandg_set_difference",  # 举例使用 wanda
    prune_data="GSM8K_cot0shot_120",  # 例如使用 GSM8K 数据
    use_diff=False,
    neg_prune=False,
    recover_from_base=False,
    p=0.5,
    q=0.5,
    top_k_heads=10,
    cache_dir="llm_weights",
    use_variant=False,
    save="temp",
    save_model=None,
    save_mask=None,
    dump_wanda_score=False,
    eval_zero_shot=True,
    eval_attack=True,
    save_attack_res=True,
    prune_part=False,
    disentangle=True,  # 注意：原 argparse 中是 --entangle_prompt_feat -> dest="disentangle", action="store_false"
    decouple_align_utility=False,
    decouple_align_misalign=False,
    rank=10,
    niter=20,
    prompt_method="cot0shot",
    dataset="GSM8K",
    role="math teacher",
    batch_size=1,
    eval_type="fixed",  # 新增参数
)


suffix = "weightonly"
cache_dir = "llm_weights"
save_dir = f"out/{args.model}/{args.sparsity_type}/{args.prune_method}_{suffix}/{args.prune_data}"
os.makedirs(save_dir, exist_ok=True)

In [4]:
# 💾 模型路径映射
modeltype2path = {
    "llama2-7b-chat-hf": "meta-llama/Llama-2-7b-chat-hf",
    "llama2-7b-hf": "meta-llama/Llama-2-7b-hf",
}


vllm_model = LLM(
            model="/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/temp/wanda_usediff_False_recover_False",
            tokenizer=modeltype2path[args.model],
            dtype="float16",
            swap_space=12,
        )

WARNING 07-26 01:54:36 config.py:398] Casting torch.bfloat16 to torch.float16.
INFO 07-26 01:54:36 llm_engine.py:72] Initializing an LLM engine with config: model='/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/temp/wanda_usediff_False_recover_False', tokenizer='meta-llama/Llama-2-7b-chat-hf', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, seed=0)
INFO 07-26 01:54:36 tokenizer.py:31] For some LLaMA V1 models, initializing the fast tokenizer may take a long time. To reduce the initialization time, consider using 'hf-internal-testing/llama-tokenizer' instead of the original tokenizer.
INFO 07-26 01:54:45 llm_engine.py:207] # GPU blocks: 879, # CPU blocks: 1536


In [5]:
import time
def apply_prompt_template(model_name, prompts):
    print(f"Applying prompt template for model: {model_name}")
    """
    根据不同的 model_name 应用聊天模板。
    支持单个字符串或字符串列表。
    """
    if isinstance(prompts, str):
        prompts = [prompts]

    formatted_prompts = []
    for prompt in prompts:
        if "llama-2" in model_name.lower():
            # LLaMA 2 Chat 模版
            B_INST, E_INST = "[INST]", "[/INST]"
            B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"
            system_prompt = "You are a helpful assistant."
            formatted = f"{B_INST} {B_SYS}{system_prompt}{E_SYS}{prompt} {E_INST}"
        elif "mistral" in model_name.lower():
            # Mistral 模版（以 <s> 和 [INST] 开头）
            formatted = f"<s>[INST] {prompt} [/INST]"
        elif "chatglm" in model_name.lower():
            # ChatGLM 模版
            formatted = f"[Round 1]\n问：{prompt}\n答："
        else:
            # 默认不加模板
            formatted = prompt

        formatted_prompts.append(formatted)

    return formatted_prompts


def _safe_generate(model, prompts, sampling_params, max_retry=5, backoff=10):
    """
    统一把 vLLM 的输出转成 List[str]。
    prompts 既可以是 str，也可以是 List[str]，最终都以 List[str] 返回。
    """
    if isinstance(prompts, str):
        prompts = [prompts]

    retry = 0
    while True:
        try:
            # vllm.LLM.generate 接收 List[str]
            raw = model.generate(prompts, sampling_params)    # type: List[RequestOutput]

            # 取每个 RequestOutput 的首个 candidate 文本
            clean = [
                (r.outputs[0].text if getattr(r, "outputs", None) else str(r)).strip()
                for r in raw
            ]
            return clean                                           # List[str]
        except Exception as e:
            retry += 1
            if retry > max_retry:
                raise RuntimeError(f"Generation failed after {max_retry} retries") from e
            wait = backoff * (2 ** (retry - 1))
            print(f"[WARN] Generate error: {e!s} | Retry {retry}/{max_retry} after {wait}s")
            time.sleep(wait)


# ---------------- ① 通用评估函数 ----------------
def evaluate_samples(
    samples: List[Dict],
    prompt_tag: str,
    vllm_model,
    sampling_params: SamplingParams,
    save_dir: Path,
    neg_prune: bool,
    sparsity_ratio: float,
    prune_data: str,
    batch_size: int,
):
    """
    通用评估函数：批量调用 vLLM，落盘 JSONL，并返回准确率。
    `samples` 每条必须含字段: input / answer
    """
    save_dir.mkdir(parents=True, exist_ok=True)
    outfile = save_dir / (
        f"gsm8k_{'top' if neg_prune else 'bottom'}_"
        f"{sparsity_ratio:.6f}_{prune_data}.jsonl"
    )

    # ---- 断点续跑 ----
    acc_list, already_done = [], 0
    if outfile.exists():
        already_done = sum(1 for _ in open(outfile))
        acc_list = [json.loads(l)["correct"] for l in open(outfile)]

    samples_iter = samples[already_done:]

    with open(outfile, "a") as fh:
        for idx in range(0, len(samples_iter), batch_size):
            chunk = samples_iter[idx : idx + batch_size]
            outputs = _safe_generate(vllm_model,
                                     [s["input"] for s in chunk],
                                     sampling_params)

            preds = [
                extract_answer(out, sample, "GSM8K")
                for out, sample in zip(outputs, chunk)
            ]

            for s, out_text, pred in zip(chunk, outputs, preds):
                correct = pred == s["answer"]
                acc_list.append(correct)
                fh.write(
                    json.dumps(
                        {
                            **s,
                            "prompt_tag": prompt_tag,
                            "output": out_text,
                            "pred": pred,
                            "correct": correct,
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
                fh.flush()

    acc = float(np.mean(acc_list)) if acc_list else 0.0
    n = len(acc_list)
    print(f"[ACC] {prompt_tag:15s}: {acc:.3%} ({int(acc*n)}/{n})")
    return {prompt_tag: acc}

def eval_gsm8k_fixed(
    args,
    vllm_model,
    tokenizer=None,
    prune_data: str = "GSM8K_direct_120",
):
    # 1) 数据文件和字段映射表
    PROMPT2DATA = {
        "GSM8K_direct":
            "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"
            "data/GSM8K/output/output.GSM8K.direct.math_teacher.llama2-7b-chat.json",
        "GSM8K_cot0shot_120":
            "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"
            "data/GSM8K/output/calibration_set_cot_120.json",
        "GSM8K_direct_120":
            "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"
            "data/GSM8K/output/calibration_set_direct_120.json",
        "GSM8K_cot0shot":
            "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"
            "data/GSM8K/output/output.GSM8K.cot0shot.goldreason.llama2-7b-chat.json",
        "GSM8K_cot0shot_goldreason":
            "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"
            "data/GSM8K/output/output.GSM8K.cot0shot.goldreason.llama2-7b-chat.json",
    }
    PROMPT2FIELD = {
        "GSM8K_direct"            : "direct.math teacher_input",
        "GSM8K_direct_120"        : "direct.math teacher_input",
        "GSM8K_cot0shot"          : "cot0shot.math teacher_input",
        "GSM8K_cot0shot_120"      : "cot0shot.math teacher_input",
        "GSM8K_cot0shot_goldreason": "cot0shot.goldreason_input",
    }

    data_file = PROMPT2DATA.get(prune_data)
    if data_file is None:
        raise ValueError(f"Unknown prompt tag: {prune_data}")
    with open(data_file, "r") as fin:
        items = json.load(fin)

    field = PROMPT2FIELD[prune_data]
    # 统一 sample 结构
    samples = [
        {
            "input"  : itm[field],
            "answer" : itm["answer"],
            "question": itm.get("question", ""),
            "id"     : itm.get("id", "")
        }
        for itm in items
    ]

    # -------- ❶  准备数据与 prompt --------
    prompt_tags = [p.strip() for p in args.prompt_method.split(",") if p.strip()]
    full_prompts = {
        tag: load_prompt(args.dataset, tag, do_role=args.role) for tag in prompt_tags
    }

    random.seed(args.seed)
    np.random.seed(args.seed)

    # 2) 提取所有 id（去掉可能为空的）
    ids = [s["id"] for s in samples if s["id"]]

    # 3) 传给 load_dataset
    data = load_dataset(
        args.dataset,
        args.nsamples,
        select_method="fixed",
        ids=ids          # 这里就是 samples 中所有 id 的列表
    )

    # -------- ❷  设置 SamplingParams --------
    sampling_params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=1024,
        n=1,              # GSM8K 评测通常一个样本即可
        stop=None,        # 统一在 extract_answer 里截断
    )

    # -------- ❸  主循环：每种 prompt 独立评估 --------
    acc_dict: Dict[str, List[bool]] = {t: [] for t in prompt_tags}

    for tag in prompt_tags:
        if args.neg_prune:
            print("Negative pruning")
            outfile = (Path(args.save)
                    / f"gsm8k_top_{args.sparsity_ratio:.6f}_{prune_data}.jsonl"
                    )
        else:
            print("Positive pruning")
            outfile = (Path(args.save)
                / f"gsm8k_bottom_{args.sparsity_ratio:.6f}_{prune_data}.jsonl"
            )

        already_done = 0
        out_fh = open(outfile, "a")

        if outfile.exists():
            # JSONL 易于续写；记录已完成行数
            already_done = sum(1 for _ in open(outfile))
            if already_done >= len(data):
                print(f"[SKIP] {outfile.name} 已完成 ({already_done}/{len(data)})")
                out_fh.close()
                acc_dict[tag] = [
                    json.loads(line)["correct"] for line in open(outfile)
                ]
                continue
            print(f"[RESUME] {outfile.name}: 已有 {already_done} 条，继续评估 …")

        dataset_iter = data[already_done:]
        dataset_chunks = [
            dataset_iter[i : i + args.batch_size]
            for i in range(0, len(dataset_iter), args.batch_size)
        ]

        for chunk in tqdm(dataset_chunks, desc=f"Eval {tag}", ncols=80):
            # 组装输入
            messages = [format_prompt(full_prompts[tag], sample) for sample in chunk]
            outputs = _safe_generate(vllm_model, messages, sampling_params)

            preds = [
                extract_answer(out_text, sample, args.dataset)
                for out_text, sample in zip(outputs, chunk)
            ]

            for sample, out_text, pred in zip(chunk, outputs, preds):
                gold = sample["answer"]
                correct = pred == gold
                acc_dict[tag].append(correct)

                record = {
                    **sample,                       # 题目 & gold answer
                    "prompt_tag": tag,
                    "input": format_prompt(full_prompts[tag], sample),
                    "output": out_text,
                    "pred": pred,
                    "gold": gold,
                    "correct": correct,
                }
                out_fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                out_fh.flush()

        out_fh.close()

    # -------- ❹  汇总指标 --------
    acc_summary = {
        tag: float(np.mean(acc)) if acc else 0.0 for tag, acc in acc_dict.items()
    }
    for tag, acc in acc_summary.items():
        n = len(acc_dict[tag])
        print(f"[ACC] {tag:15s}: {acc:.3%} ({int(acc*n)}/{n})")

    return acc_summary



In [6]:

def eval_gsm8k_random(
    args,
    vllm_model,
    tokenizer=None,
    prune_data: str = "GSM8K_direct_120",
):
    """
    Evaluate a (possibly pruned) model on GSM8K.

    Parameters
    ----------
    args : argparse.Namespace
        应包含 dataset, prompts, role, seed, nsamples, batch_size,
        temperature, top_p, max_new_tokens, model_name, sparsity_ratio 等字段
    vllm_model : LLM
        已包装好的 vLLM 模型实例，需暴露 generate / batch_generate 接口
    tokenizer : transformers.PreTrainedTokenizer, optional
        仅在需要自定义特殊 token 时使用
    prune_data : str, default "GSM8K_direct_120"
        记录用的是哪一版 pruned 数据，可写入结果文件名方便区分
    save_filepath : str
        结果输出目录。每个 prompt 对应一个 *.jsonl* 结果文件

    Returns
    -------
    Dict[str, float]
        {prompt_tag: accuracy}
    """
    
    # -------- ❶  准备数据与 prompt --------
    prompt_tags = [p.strip() for p in args.prompt_method.split(",") if p.strip()]
    full_prompts = {
        tag: load_prompt(args.dataset, tag, do_role=args.role) for tag in prompt_tags
    }

    random.seed(args.seed)
    np.random.seed(args.seed)
    data = load_dataset(args.dataset, args.nsamples)

    # -------- ❷  设置 SamplingParams --------
    sampling_params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=1024,
        n=1,              # GSM8K 评测通常一个样本即可
        stop=None,        # 统一在 extract_answer 里截断
    )

    # -------- ❸  主循环：每种 prompt 独立评估 --------
    acc_dict: Dict[str, List[bool]] = {t: [] for t in prompt_tags}

    for tag in prompt_tags:
        if args.neg_prune:
            print("Negative pruning")
            outfile = (Path(args.save)
                    / f"gsm8k_top_{args.sparsity_ratio:.6f}_{prune_data}.jsonl"
                    )
        else:
            print("Positive pruning")
            outfile = (Path(args.save)
                / f"gsm8k_bottom_{args.sparsity_ratio:.6f}_{prune_data}.jsonl"
            )

        already_done = 0
        out_fh = open(outfile, "a")

        if outfile.exists():
            # JSONL 易于续写；记录已完成行数
            already_done = sum(1 for _ in open(outfile))
            if already_done >= len(data):
                print(f"[SKIP] {outfile.name} 已完成 ({already_done}/{len(data)})")
                out_fh.close()
                acc_dict[tag] = [
                    json.loads(line)["correct"] for line in open(outfile)
                ]
                continue
            print(f"[RESUME] {outfile.name}: 已有 {already_done} 条，继续评估 …")

        dataset_iter = data[already_done:]
        dataset_chunks = [
            dataset_iter[i : i + args.batch_size]
            for i in range(0, len(dataset_iter), args.batch_size)
        ]

        for chunk in tqdm(dataset_chunks, desc=f"Eval {tag}", ncols=80):
            # 组装输入
            messages = [format_prompt(full_prompts[tag], sample) for sample in chunk]
            outputs = _safe_generate(vllm_model, messages, sampling_params)

            preds = [
                extract_answer(out_text, sample, args.dataset)
                for out_text, sample in zip(outputs, chunk)
            ]

            for sample, out_text, pred in zip(chunk, outputs, preds):
                gold = sample["answer"]
                correct = pred == gold
                acc_dict[tag].append(correct)

                record = {
                    **sample,                       # 题目 & gold answer
                    "prompt_tag": tag,
                    "input": format_prompt(full_prompts[tag], sample),
                    "output": out_text,
                    "pred": pred,
                    "gold": gold,
                    "correct": correct,
                }
                out_fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                out_fh.flush()

        out_fh.close()

    # -------- ❹  汇总指标 --------
    acc_summary = {
        tag: float(np.mean(acc)) if acc else 0.0 for tag, acc in acc_dict.items()
    }
    for tag, acc in acc_summary.items():
        n = len(acc_dict[tag])
        print(f"[ACC] {tag:15s}: {acc:.3%} ({int(acc*n)}/{n})")

    return acc_summary

In [ ]:
save_filepath = os.path.join(args.save, f"log_{args.prune_method}.txt")

print(f"Evaluating GSM8K {args.prune_data} with {args.model}")
score = eval_gsm8k_fixed(
    args,
    vllm_model,
    None,
    prune_data=args.prune_data,
)



Evaluating GSM8K GSM8K_cot0shot_120 with llama2-7b-chat-hf
Positive pruning
[RESUME] gsm8k_bottom_0.500000_GSM8K_cot0shot_120.jsonl: 已有 0 条，继续评估 …


Eval cot0shot:   0%|                                      | 0/2 [00:00<?, ?it/s]

Eval cot0shot:  50%|███████████████               | 1/2 [00:29<00:29, 29.58s/it]

extract_answer: list index out of range


Eval cot0shot: 100%|██████████████████████████████| 2/2 [00:59<00:00, 29.57s/it]

extract_answer: list index out of range
[ACC] cot0shot       : 0.000% (0/2)


: 